# 01 - Ingestão dos Dados

**Fonte**: Brazilian E-Commerce Public Dataset by Olist - https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
(licença CC BY-NC-SA 4.0, uso educacional).

Para que a ingestão seja **reproduzível por código sem exigir credenciais de API do Kaggle**, este notebook baixa os
mesmos 9 arquivos CSV originais a partir de um mirror público no GitHub
(`ckoliveiraa/pipeline-olist`), que replica os arquivos do dataset oficial com os mesmos nomes e conteúdo
(comprimidos em `.gz`). Os arquivos são descompactados e salvos em `data/raw/`, que nunca é editado manualmente.

Ver `docs/02_fontes_de_dados.md` para detalhes de cada arquivo.

In [1]:
import gzip
import shutil
import urllib.request
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/ckoliveiraa/pipeline-olist/main/raw"

ARQUIVOS = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_orders_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation",
]

In [2]:
def baixar_e_descomprimir(nome: str) -> Path:
    """Baixa <nome>.csv.gz do mirror e salva <nome>.csv em data/raw, se ainda não existir."""
    destino_csv = RAW_DIR / f"{nome}.csv"
    if destino_csv.exists():
        return destino_csv

    url = f"{BASE_URL}/{nome}.csv.gz"
    destino_gz = RAW_DIR / f"{nome}.csv.gz"
    urllib.request.urlretrieve(url, destino_gz)

    with gzip.open(destino_gz, "rb") as f_in, open(destino_csv, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    destino_gz.unlink()
    return destino_csv


caminhos = {nome: baixar_e_descomprimir(nome) for nome in ARQUIVOS}
caminhos

{'olist_customers_dataset': WindowsPath('../data/raw/olist_customers_dataset.csv'),
 'olist_geolocation_dataset': WindowsPath('../data/raw/olist_geolocation_dataset.csv'),
 'olist_order_items_dataset': WindowsPath('../data/raw/olist_order_items_dataset.csv'),
 'olist_order_payments_dataset': WindowsPath('../data/raw/olist_order_payments_dataset.csv'),
 'olist_order_reviews_dataset': WindowsPath('../data/raw/olist_order_reviews_dataset.csv'),
 'olist_orders_dataset': WindowsPath('../data/raw/olist_orders_dataset.csv'),
 'olist_products_dataset': WindowsPath('../data/raw/olist_products_dataset.csv'),
 'olist_sellers_dataset': WindowsPath('../data/raw/olist_sellers_dataset.csv'),
 'product_category_name_translation': WindowsPath('../data/raw/product_category_name_translation.csv')}

## Carregamento e verificação da ingestão

In [3]:
dados = {nome: pd.read_csv(caminho) for nome, caminho in caminhos.items()}

resumo = pd.DataFrame(
    [(nome, df.shape[0], df.shape[1]) for nome, df in dados.items()],
    columns=["arquivo", "linhas", "colunas"],
)
resumo

,arquivo,linhas,colunas
0,olist_customers_dataset,99441,5
1,olist_geolocation_dataset,1000163,5
2,olist_order_items_dataset,112650,7
3,olist_order_payments_dataset,103886,5
4,olist_order_reviews_dataset,99224,7
5,olist_orders_dataset,99441,8
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


A ingestão carregou os 9 arquivos com sucesso. As contagens de linhas conferem com as estatísticas públicas
conhecidas do dataset (ex.: ~99.441 pedidos, ~1.000.163 registros de geolocalização), confirmando a integridade
do download.

Próxima etapa: `02_avaliacao_qualidade.ipynb`, que carrega os mesmos arquivos de `data/raw/` para investigar
problemas de qualidade.